# EDA Dataset Netflix — Panduan Lengkap

**Sub-CPMK:** **P4** — EDA univariat/bivariat; visualisasi dengan Matplotlib & Seaborn.

Notebook ini adalah **tutorial mandiri** (melengkapi `minggu_05.ipynb`): eksplorasi metadata katalog Netflix (~2019), tren waktu, perbandingan **Movie** vs **TV Show**, dan ringkasan insight untuk analitik lanjutan.

**Konteks data:** [Kaggle — Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows) (~**8807** judul). Bukan dataset klasifikasi dengan satu target biner; variabel pemisah utama: **`type`** dan dimensi waktu (`release_year`, `year_added`).

**Sumber data:** mirror [TidyTuesday 2021-04-20](https://github.com/rfordatascience/tidytuesday/blob/master/data/2021/2021-04-20/readme.md) via URL GitHub raw. **Unduhan pertama membutuhkan internet.** Alternatif offline: salin `netflix_titles.csv` ke folder `data/` dan ganti baris `read_csv`.

**Pertanyaan analitik (diadaptasi dari tutorial cr1deg0 / Analytics Vidhya):**
- (Q1) Bagaimana pertumbuhan judul yang ditambahkan ke Netflix per tahun?
- (Q2–Q3) Negara produksi teratas untuk **Movie** vs **TV Show**?
- (Q4) Bagaimana distribusi **rating** menurut tipe konten?
- (Q5) Apakah ada pola musiman pada bulan penambahan (`month_added`)?
- (Q6) Berapa lama antara tahun rilis dan tahun masuk katalog (`years_on_platform`)?
- (Q7) Bagaimana perbedaan **durasi** film (menit) vs serial (season)?
- (Q8) Genre apa yang paling sering muncul di kolom `listed_in`?

**Referensi:**
- [Analytics Vidhya — Visualizing Netflix Data](https://www.analyticsvidhya.com/blog/2021/07/visualizing-netflix-data-using-python/)
- [Medium — Netflix EDA and Visualization](https://medium.com/analytics-vidhya/netflix-movies-and-tvshows-exploratory-data-analysis-eda-and-visualization-using-python-80753fcfcf7)
- [cr1deg0 — Netflix dataset analysis](https://cr1deg0.github.io/Netflix_dataset_analysis/)
- [DEV — Beginner Netflix analysis](https://dev.to/kody_c2fc16bba41e453db5da/beginner-friendly-netflix-data-analysis-project-using-python-full-guide-video-5een)
- [TidyTuesday readme](https://github.com/rfordatascience/tidytuesday/blob/master/data/2021/2021-04-20/readme.md)
- Modul PDF: `modul-05.tex` (Minggu 5 praktikum)


## 0. Kerangka CRISP-DM dan kamus fitur

Fase **Data Understanding** (CRISP-DM): pahami arti kolom sebelum memplot. EDA **iteratif** — temuan mengarahkan cleaning (`minggu_03`), encoding (`minggu_04`), dan pemodelan (`minggu_06`–`minggu_07`).

| Kolom | Arti singkat |
|-------|----------------|
| `show_id` | ID unik judul di katalog |
| `type` | `Movie` atau `TV Show` |
| `title` | Judul tayangan |
| `director` | Sutradara (banyak NA pada serial) |
| `cast` | Pemeran utama |
| `country` | Negara produksi (bisa multi-negara, dipisah koma) |
| `date_added` | Tanggal judul masuk katalog Netflix |
| `release_year` | Tahun rilis produksi asli |
| `rating` | Klasifikasi usia (TV-MA, PG-13, …) |
| `duration` | Menit (film) atau jumlah season (serial) |
| `listed_in` | Genre/kategori (string multi-label, dipisah koma) |
| `description` | Sinopsis singkat |

**Variabel turunan (dibuat di §2b):** `year_added`, `month_added`, `duration_int`, `is_season`, `years_on_platform`, `primary_country`, `primary_genre`.


## 1. Persiapan lingkungan

Import pustaka standar praktikum; muat CSV dari URL TidyTuesday.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
NETFLIX_URL = (
    "https://raw.githubusercontent.com/rfordatascience/tidytuesday/"
    "main/data/2021/2021-04-20/netflix_titles.csv"
)

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

**Memuat data.** Ganti `NETFLIX_URL` dengan path lokal `data/netflix_titles.csv` jika lab tanpa internet.


In [ ]:
print("Mengunduh/memuat netflix_titles.csv (internet pada unduhan pertama)...")
df = pd.read_csv(NETFLIX_URL)
df = df.copy()
print("Bentuk data:", df.shape)
df.head()

**Cuplikan akhir, info, dan statistik deskriptif.**


In [ ]:
display(df.tail())
df.info()

In [ ]:
display(df.describe())
display(df.describe(include="object"))

### 1b. Audit kualitas data

`sample` untuk inspeksi acak; cek duplikat pada `show_id` dan baris penuh.


In [ ]:
print("Cuplikan acak 20 judul:")
display(df.sample(20, random_state=RANDOM_STATE))

print("\nDuplikat show_id:", df["show_id"].duplicated().sum())
print("Duplikat baris penuh:", df.duplicated().sum())
print("\nNilai unik type:", df["type"].unique())

**Interpretasi (audit):** Setiap baris satu judul unik (`show_id`); dataset berisi dua tipe konten utama. Tidak ada duplikat baris penuh pada versi TidyTuesday ini.


## 2. Missing values dan pola NA

Menghitung NA per kolom sebelum visual. Kolom `director` sering kosong karena banyak serial tanpa satu sutradara tercatat.


In [ ]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_count / len(df) * 100).round(2)
missing_tbl = pd.DataFrame({"jumlah_na": missing_count, "persen": missing_pct})
missing_tbl[missing_tbl["jumlah_na"] > 0]

**Barplot dan heatmap missing.** Alternatif `missingno` tanpa dependency tambahan.


In [ ]:
miss = missing_tbl[missing_tbl["jumlah_na"] > 0].sort_values("jumlah_na", ascending=True)
plt.figure(figsize=(8, 4))
sns.barplot(x=miss["jumlah_na"], y=miss.index, hue=miss.index, palette="Reds_d", legend=False)
plt.title("Jumlah nilai hilang (NA) per kolom")
plt.xlabel("Jumlah NA")
plt.ylabel("Kolom")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap="magma")
plt.title("Pola NA per kolom (setiap baris = satu judul)")
plt.xlabel("Kolom")
plt.tight_layout()
plt.show()

**Interpretasi (missing):** `director` ~30% NA; `cast`/`country` ~9%; `date_added`/`rating`/`duration` hampir lengkap. Analisis sutradara perlu imputasi atau filter; eksplorasi tren waktu membutuhkan `date_added` valid.


## 2b. Wrangling ringan untuk EDA

Transformasi minimal agar plot numerik/waktu bermakna. **Bukan** pipeline cleaning penuh (`minggu_03.ipynb`).


In [ ]:
df_eda = df.copy()

# Parse tanggal penambahan ke katalog
df_eda["date_added"] = pd.to_datetime(df_eda["date_added"], format="%B %d, %Y", errors="coerce")
df_eda["year_added"] = df_eda["date_added"].dt.year
df_eda["month_added"] = df_eda["date_added"].dt.month

# Durasi: menit vs jumlah season
df_eda["is_season"] = df_eda["duration"].str.contains("Season", case=False, na=False)
dur_str = df_eda["duration"].astype(str)
dur_str = dur_str.str.replace(" min", "", regex=False)
dur_str = dur_str.str.replace(" Seasons", "", regex=False)
dur_str = dur_str.str.replace(" Season", "", regex=False)
df_eda["duration_int"] = pd.to_numeric(dur_str, errors="coerce")

# Lag rilis → masuk katalog
df_eda["years_on_platform"] = df_eda["year_added"] - df_eda["release_year"]

# Negara & genre pertama (simplifikasi multi-label)
df_eda["primary_country"] = df_eda["country"].fillna("Unknown").str.split(",").str[0].str.strip()
df_eda["primary_genre"] = df_eda["listed_in"].str.split(",").str[0].str.strip()

# Subset analisis: baris dengan date/rating/duration lengkap
n_before = len(df_eda)
df_clean = df_eda.dropna(subset=["date_added", "rating", "duration"]).copy()
print(f"Baris untuk analisis terarah: {len(df_clean)} dari {n_before} "
      f"({n_before - len(df_clean)} dihapus karena NA kritis)")
df_clean.head(2)

**Interpretasi (wrangling):** `duration_int` = menit (film) atau jumlah season (serial); jangan bandingkan langsung tanpa memisah `type`. `years_on_platform` negatif atau sangat besar perlu disaring saat modeling.


## 3. EDA tabular

Ringkasan frekuensi dan agregat per `type` sebelum visual.


In [ ]:
print("Distribusi type:")
print(df["type"].value_counts())
print("\nTop 10 rating:")
print(df["rating"].value_counts().head(10))

print("\nAgregat numerik per type (subset df_clean):")
display(
    df_clean.groupby("type")[["release_year", "duration_int", "years_on_platform"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

ct = pd.crosstab(df_clean["type"], df_clean["rating"], margins=True)
print("\nCrosstab type × rating (jumlah):")
display(ct)

**Interpretasi (tabel):** TV Show sedikit lebih banyak daripada Movie pada snapshot ini; rating TV-MA dan TV-14 dominan. Median `duration_int` jauh lebih kecil untuk Movie (menit) vs TV Show (season).


## 4. EDA univariat

Distribusi marginal variabel utama.


**Proporsi Movie vs TV Show.**


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="type", order=df["type"].value_counts().index)
plt.title("Jumlah judul menurut type")
plt.xlabel("type")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.show()

**Interpretasi:** Katalog sedikit condong ke **TV Show**; pemisahan `type` relevan untuk hampir semua plot bivariat berikutnya.


**Distribusi tahun rilis (`release_year`).**


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["release_year"], bins=30, kde=True, color="steelblue")
plt.title("Distribusi release_year")
plt.xlabel("release_year")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

**Interpretasi:** Massa distribusi pada 2010-an; ekor kiri judul klasik lebih jarang.


**Rating usia — frekuensi.**


In [ ]:
rating_order = df["rating"].value_counts().index
plt.figure(figsize=(9, 4))
sns.countplot(data=df, y="rating", order=rating_order)
plt.title("Frekuensi rating")
plt.xlabel("Jumlah")
plt.ylabel("rating")
plt.tight_layout()
plt.show()

**Q1 — Pertumbuhan judul ditambahkan per tahun (`year_added`).**


In [ ]:
added_yearly = df_clean["year_added"].value_counts().sort_index()
plt.figure(figsize=(10, 4))
plt.plot(added_yearly.index, added_yearly.values, marker="o")
plt.title("Jumlah judul ditambahkan ke Netflix per tahun")
plt.xlabel("year_added")
plt.ylabel("Jumlah judul")
plt.tight_layout()
plt.show()

**Interpretasi (Q1):** Lonjakan penambahan sejak ~2017; data awal 2007–2015 sangat sedikit (lisensi & peluncuhan platform streaming).


**Q5 — Pola musiman: bulan penambahan (`month_added`).**


In [ ]:
month_order = range(1, 13)
month_counts = df_clean["month_added"].value_counts().reindex(month_order, fill_value=0)
plt.figure(figsize=(8, 4))
sns.barplot(x=month_counts.index, y=month_counts.values, palette="viridis")
plt.title("Jumlah penambahan judul per bulan")
plt.xlabel("month_added (1=Jan)")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.show()

**Interpretasi (Q5):** Beberapa bulan sedikit lebih ramai (mis. akhir tahun); pola musiman lemah dibanding tren tahunan.


## 5. EDA bivariat — Movie vs TV Show

Perbandingan konten film dan serial.


**Q4 — Distribusi rating menurut type.**


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, x="rating", hue="type", order=df_clean["rating"].value_counts().index)
plt.title("Rating menurut type (Movie vs TV Show)")
plt.xlabel("rating")
plt.ylabel("Jumlah")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Interpretasi (Q4):** TV Show dominan pada TV-MA/TV-14; Movie lebih beragam pada PG-13 dan R.


**Q7 — Durasi: boxplot & violin `duration_int` × type.**


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df_clean, x="type", y="duration_int", ax=ax[0])
ax[0].set_title("Boxplot duration_int per type")
ax[0].set_xlabel("type")
ax[0].set_ylabel("duration_int (menit atau season)")
sns.violinplot(data=df_clean, x="type", y="duration_int", ax=ax[1])
ax[1].set_title("Violinplot duration_int per type")
plt.tight_layout()
plt.show()

**Interpretasi (Q7):** Movie ~90 menit median; TV Show ~1–2 season — skala berbeda, jangan gabungkan tanpa `type`.


**Tahun rilis menurut type.**


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df_clean, x="release_year", hue="type", bins=25, kde=False, multiple="stack")
plt.title("release_year menurut type")
plt.xlabel("release_year")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

**Q6 — Lag tahun: `years_on_platform` menurut type.**


In [ ]:
lag = df_clean[df_clean["years_on_platform"].between(0, 80)]
plt.figure(figsize=(8, 4))
sns.boxplot(data=lag, x="type", y="years_on_platform")
plt.title("years_on_platform (year_added - release_year)")
plt.xlabel("type")
plt.ylabel("Tahun")
plt.tight_layout()
plt.show()

**Interpretasi (Q6):** Movie sering masuk katalog beberapa tahun setelah rilis bioskop; TV Show bervariasi (produksi lama vs serial baru).


**Q2 — Top 15 negara produksi (Movie).**


In [ ]:
movies = df_clean[df_clean["type"] == "Movie"]
top_movie_cty = movies["primary_country"].value_counts().head(15)
plt.figure(figsize=(8, 5))
sns.barplot(x=top_movie_cty.values, y=top_movie_cty.index, hue=top_movie_cty.index, palette="Blues_d", legend=False)
plt.title("Top 15 negara — Movie (primary_country)")
plt.xlabel("Jumlah")
plt.ylabel("Negara")
plt.tight_layout()
plt.show()

**Q3 — Top 15 negara produksi (TV Show).**


In [ ]:
tv = df_clean[df_clean["type"] == "TV Show"]
top_tv_cty = tv["primary_country"].value_counts().head(15)
plt.figure(figsize=(8, 5))
sns.barplot(x=top_tv_cty.values, y=top_tv_cty.index, hue=top_tv_cty.index, palette="Greens_d", legend=False)
plt.title("Top 15 negara — TV Show (primary_country)")
plt.xlabel("Jumlah")
plt.ylabel("Negara")
plt.tight_layout()
plt.show()

**Interpretasi (Q2–Q3):** **United States** dominan pada keduanya; TV Show lebih internasional (UK, Japan, South Korea) dibanding Movie (India ketiga).


**Heatmap jumlah: type × rating (subset rating utama).**


In [ ]:
top_ratings = df_clean["rating"].value_counts().head(8).index
sub = df_clean[df_clean["rating"].isin(top_ratings)]
pivot = pd.crosstab(sub["type"], sub["rating"])
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd")
plt.title("Jumlah judul: type × rating (top 8 rating)")
plt.xlabel("rating")
plt.ylabel("type")
plt.tight_layout()
plt.show()

## 6. EDA multivariat

Korelasi numerik dan pasangan fitur.


In [ ]:
num_cols = ["release_year", "year_added", "month_added", "duration_int", "years_on_platform"]
num = df_clean[num_cols].dropna()
display(num.corr().round(3))

plt.figure(figsize=(6, 5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Korelasi fitur numerik (subset df_clean)")
plt.tight_layout()
plt.show()

**Pairplot subset dengan hue=type.**


In [ ]:
pair_cols = ["release_year", "duration_int", "years_on_platform"]
sns.pairplot(
    df_clean[pair_cols + ["type"]].dropna(),
    hue="type",
    corner=True,
    plot_kws={"alpha": 0.4},
)
plt.show()

**Cuplikan: korelasi setelah one-hot (menuju minggu_04).**


In [ ]:
top_r = df_clean["rating"].value_counts().head(6).index
sub_enc = df_clean[df_clean["rating"].isin(top_r)][["type", "rating"]].copy()
dummies = pd.get_dummies(sub_enc, drop_first=True)
if len(dummies.columns) > 1:
  corr_d = dummies.corr()
  plt.figure(figsize=(7, 5))
  sns.heatmap(corr_d, annot=True, fmt=".2f", cmap="coolwarm", center=0)
  plt.title("Korelasi — type + rating (top 6, get_dummies)")
  plt.tight_layout()
  plt.show()

**Interpretasi (multivariat):** `year_added` dan `release_year` berkorelasi moderat; `duration_int` tidak linear terhadap tahun. Korelasi bukan kausalitas.


## 7. Eksplorasi genre dan negara (teks)

**Q8 — Top 15 genre** dari `listed_in` (split koma, hitung per label).


In [ ]:
genre_series = (
    df["listed_in"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)
top_genres = genre_series.value_counts().head(15)
plt.figure(figsize=(8, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index, palette="Purples_d", legend=False)
plt.title("Top 15 genre (listed_in, per label)")
plt.xlabel("Jumlah kemunculan")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()
print(top_genres)

**Top 10 primary_country** (negara pertama pada string multi-negara).


In [ ]:
top_cty = df_eda["primary_country"].value_counts().head(10)
display(top_cty)

**Interpretasi (Q8):** **International Movies**, **Dramas**, **Comedies** sering muncul; satu judul bisa masuk banyak genre sehingga total kemunculan > jumlah baris.


## 8. Ringkasan temuan, bias, dan hipotesis

### Ringkasan temuan (EDA)

1. **Skala:** ~8807 judul; dominasi **TV Show** sedikit di atas Movie.
2. **Waktu (Q1):** Penambahan katalog melonjak sejak **2017**; data tahun awal platform jarang.
3. **Geografi (Q2–Q3):** **United States** terbesar; TV lebih internasional (UK, Japan, Korea) vs Movie (India kuat).
4. **Rating (Q4):** TV-MA/TV-14 dominan serial; film lebih bervariasi PG-13/R.
5. **Durasi (Q7):** Film ~menit; serial ~season — pisahkan `type` sebelum statistik.
6. **Novelty (Q6):** `years_on_platform` bervariasi; film cenderung masuk beberapa tahun setelah rilis.
7. **Musiman (Q5):** Pola bulan lemah dibanding tren tahunan.
8. **Genre (Q8):** Multi-label; Dramas/International Movies puncak frekuensi label.

### Bias dan limitasi

- Snapshot **2019**; judul keluar-masuk karena lisensi tidak tercermin.
- `primary_country` / `primary_genre` hanya label **pertama** — menyederhanakan co-production.
- **~30%** `director` kosong — bias pada analisis sutradara.
- Netflix ≠ seluruh industri streaming; generalisasi terbatas.

### Hipoteses lanjutan (tanpa implementasi di notebook ini)

- **Klasifikasi** `type` dari `duration_int`, `rating`, dan `release_year`.
- **Clustering** genre atau negara untuk segmentasi katalog.
- **Regresi** jumlah penambahan per kuartal terhadap fitur eksternal (di luar dataset).

### Langkah berikutnya (alur praktikum)

1. **Cleaning & imputasi** — `minggu_03.ipynb`
2. **Encoding & scaling** — `minggu_04.ipynb`
3. **Pemodelan** — `minggu_06.ipynb` (klasifikasi) / `minggu_07.ipynb` (regresi jika target kontinu dibuat)

Jalankan **Kernel → Restart & Run All** untuk memastikan urutan sel konsisten.
